<a href="https://colab.research.google.com/github/Brianoyaro/ml_zoomcamp/blob/main/pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/alexeygrigorev/clothing-dataset-small.git

Cloning into 'clothing-dataset-small'...
remote: Enumerating objects: 3839, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (400/400), done.
remote: Total 3839 (delta 9), reused 385 (delta 0), pack-reused 3439 (from 1)
Receiving objects: 100% (3839/3839), 100.58 MiB | 19.41 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Updating files: 100% (3783/3783), done.


In [2]:
import os
from torch.utils.data import Dataset
from PIL import Image

# A python dataset for image loading
class ClothingDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
# Simple pre-processing

from torchvision import transforms

input_size = 224

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [4]:
# Data Loader

from torch.utils.data import DataLoader

train_dataset = ClothingDataset(
    data_dir='./clothing-dataset-small/train',
    transform=train_transforms
)

val_dataset = ClothingDataset(
    data_dir='./clothing-dataset-small/validation',
    transform=val_transforms
)

print(f"train dataset object: {train_dataset}")
print(f"val dataset object: {val_dataset}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

train dataset object: <__main__.ClothingDataset object at 0x799ddaee5940>
val dataset object: <__main__.ClothingDataset object at 0x799ddaea39d0>


In [5]:
# Build the model

import torch.nn as nn
import torchvision.models as models

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.output_layer = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.output_layer(x)
        return x

In [6]:
import torch
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ClothingClassifierMobileNet(num_classes=10)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 146MB/s]


In [7]:
'''# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Calculate the loss
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss += loss.item()
        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        # Update total and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # Disable gradient calculation for validation
    with torch.no_grad():
        # Iterate over the validation data
        for inputs, labels in val_loader:
            # Move data to the specified device (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            # Calculate the loss
            loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_loss += loss.item()
            # Get predictions
            _, predicted = torch.max(outputs.data, 1)
            # Update total and correct predictions
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')'''

"# Training loop\nnum_epochs = 10\n\nfor epoch in range(num_epochs):\n    # Training phase\n    model.train()  # Set the model to training mode\n    running_loss = 0.0\n    correct = 0\n    total = 0\n\n    # Iterate over the training data\n    for inputs, labels in train_loader:\n        # Move data to the specified device (GPU or CPU)\n        inputs, labels = inputs.to(device), labels.to(device)\n\n        # Zero the parameter gradients to prevent accumulation\n        optimizer.zero_grad()\n        # Forward pass\n        outputs = model(inputs)\n        # Calculate the loss\n        loss = criterion(outputs, labels)\n        # Backward pass and optimize\n        loss.backward()\n        optimizer.step()\n\n        # Accumulate training loss\n        running_loss += loss.item()\n        # Get predictions\n        _, predicted = torch.max(outputs.data, 1)\n        # Update total and correct predictions\n        total += labels.size(0)\n        correct += (predicted == labels).sum().

In [8]:
def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

In [9]:
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs=10, device=device)

Epoch 1/10
  Train Loss: 1.5130, Train Acc: 0.6193
  Val Loss: 0.7248, Val Acc: 0.7859
Epoch 2/10
  Train Loss: 0.8223, Train Acc: 0.7520
  Val Loss: 0.7949, Val Acc: 0.7742
Epoch 3/10
  Train Loss: 0.6580, Train Acc: 0.8028
  Val Loss: 0.7425, Val Acc: 0.7889
Epoch 4/10
  Train Loss: 0.7090, Train Acc: 0.7930
  Val Loss: 1.1142, Val Acc: 0.7361
Epoch 5/10
  Train Loss: 0.5205, Train Acc: 0.8481
  Val Loss: 1.1034, Val Acc: 0.7302
Epoch 6/10
  Train Loss: 0.5793, Train Acc: 0.8351
  Val Loss: 0.9659, Val Acc: 0.7595
Epoch 7/10
  Train Loss: 0.4905, Train Acc: 0.8520
  Val Loss: 1.2250, Val Acc: 0.7126
Epoch 8/10
  Train Loss: 0.5473, Train Acc: 0.8449
  Val Loss: 0.9077, Val Acc: 0.7713
Epoch 9/10
  Train Loss: 0.5403, Train Acc: 0.8517
  Val Loss: 1.1282, Val Acc: 0.7713
Epoch 10/10
  Train Loss: 0.5034, Train Acc: 0.8553
  Val Loss: 1.0732, Val Acc: 0.7654


In [10]:
# Tunig the learning rate

def make_model(learning_rate=0.01):
    model = ClothingClassifierMobileNet(num_classes=10)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

learning_rates = [0.0001, 0.001, 0.01, 0.1]
num_epochs = 10

for lr in learning_rates:
    print(f'\n=== Learning Rate: {lr} ===')
    model, optimizer = make_model(learning_rate=lr)
    train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)


=== Learning Rate: 0.0001 ===
Epoch 1/10
  Train Loss: 1.9917, Train Acc: 0.3100
  Val Loss: 1.7698, Val Acc: 0.3842
Epoch 2/10
  Train Loss: 1.6317, Train Acc: 0.4889
  Val Loss: 1.4849, Val Acc: 0.5455
Epoch 3/10
  Train Loss: 1.4012, Train Acc: 0.5828
  Val Loss: 1.2910, Val Acc: 0.5806
Epoch 4/10
  Train Loss: 1.2439, Train Acc: 0.6450
  Val Loss: 1.1608, Val Acc: 0.6510
Epoch 5/10
  Train Loss: 1.1236, Train Acc: 0.6789
  Val Loss: 1.0556, Val Acc: 0.6921
Epoch 6/10
  Train Loss: 1.0392, Train Acc: 0.7109
  Val Loss: 0.9851, Val Acc: 0.7185
Epoch 7/10
  Train Loss: 0.9735, Train Acc: 0.7311
  Val Loss: 0.9330, Val Acc: 0.7185
Epoch 8/10
  Train Loss: 0.9063, Train Acc: 0.7503
  Val Loss: 0.8861, Val Acc: 0.7419
Epoch 9/10
  Train Loss: 0.8609, Train Acc: 0.7696
  Val Loss: 0.8419, Val Acc: 0.7449
Epoch 10/10
  Train Loss: 0.8192, Train Acc: 0.7790
  Val Loss: 0.8104, Val Acc: 0.7625

=== Learning Rate: 0.001 ===
Epoch 1/10
  Train Loss: 1.2996, Train Acc: 0.5789
  Val Loss: 0.849

In [11]:
best_learning_rate = 0.001

In [12]:
# model checkpointing - saving the best model while training

def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    best_val_accuracy = 0.0  # Initialize variable to track the best validation accuracy

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        # Checkpoint the model if validation accuracy improved
        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            checkpoint_path = f'mobilenet_v2_{epoch+1:02d}_{val_acc:.3f}.pth'
            torch.save(model.state_dict(), checkpoint_path)
            print(f'Checkpoint saved: {checkpoint_path}')

In [13]:
# training model with checkpoint enabled

model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

Epoch 1/10
  Train Loss: 1.3175, Train Acc: 0.5870
  Val Loss: 0.8296, Val Acc: 0.7625
Checkpoint saved: mobilenet_v2_01_0.762.pth
Epoch 2/10
  Train Loss: 0.7637, Train Acc: 0.7604
  Val Loss: 0.6953, Val Acc: 0.7507
Epoch 3/10
  Train Loss: 0.6350, Train Acc: 0.8002
  Val Loss: 0.6505, Val Acc: 0.7683
Checkpoint saved: mobilenet_v2_03_0.768.pth
Epoch 4/10
  Train Loss: 0.5588, Train Acc: 0.8250
  Val Loss: 0.5929, Val Acc: 0.8065
Checkpoint saved: mobilenet_v2_04_0.806.pth
Epoch 5/10
  Train Loss: 0.5216, Train Acc: 0.8266
  Val Loss: 0.6315, Val Acc: 0.7918
Epoch 6/10
  Train Loss: 0.4666, Train Acc: 0.8530
  Val Loss: 0.5777, Val Acc: 0.8094
Checkpoint saved: mobilenet_v2_06_0.809.pth
Epoch 7/10
  Train Loss: 0.4407, Train Acc: 0.8527
  Val Loss: 0.5485, Val Acc: 0.7977
Epoch 8/10
  Train Loss: 0.4218, Train Acc: 0.8634
  Val Loss: 0.5619, Val Acc: 0.8123
Checkpoint saved: mobilenet_v2_08_0.812.pth
Epoch 9/10
  Train Loss: 0.4031, Train Acc: 0.8719
  Val Loss: 0.5552, Val Acc: 0.80

In [14]:
# Adding inner layers

import torch.nn as nn
import torchvision.models as models

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU()

        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)

        x = self.inner(x)
        x = self.relu(x)

        x = self.output_layer(x)
        return x



def make_model(learning_rate=0.001, size_inner=100):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer


model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

Epoch 1/10
  Train Loss: 1.2532, Train Acc: 0.5997
  Val Loss: 0.8023, Val Acc: 0.7331
Checkpoint saved: mobilenet_v2_01_0.733.pth
Epoch 2/10
  Train Loss: 0.6807, Train Acc: 0.7800
  Val Loss: 0.6000, Val Acc: 0.7918
Checkpoint saved: mobilenet_v2_02_0.792.pth
Epoch 3/10
  Train Loss: 0.5604, Train Acc: 0.8132
  Val Loss: 0.5948, Val Acc: 0.7889
Epoch 4/10
  Train Loss: 0.5224, Train Acc: 0.8201
  Val Loss: 0.5769, Val Acc: 0.8035
Checkpoint saved: mobilenet_v2_04_0.804.pth
Epoch 5/10
  Train Loss: 0.4551, Train Acc: 0.8413
  Val Loss: 0.5793, Val Acc: 0.7830
Epoch 6/10
  Train Loss: 0.4156, Train Acc: 0.8638
  Val Loss: 0.5404, Val Acc: 0.8182
Checkpoint saved: mobilenet_v2_06_0.818.pth
Epoch 7/10
  Train Loss: 0.3716, Train Acc: 0.8735
  Val Loss: 0.5841, Val Acc: 0.7889
Epoch 8/10
  Train Loss: 0.3680, Train Acc: 0.8709
  Val Loss: 0.6496, Val Acc: 0.7859
Epoch 9/10
  Train Loss: 0.3545, Train Acc: 0.8752
  Val Loss: 0.6647, Val Acc: 0.7595
Epoch 10/10
  Train Loss: 0.3373, Train A

In [15]:
# Drop-out and regularization


import torch.nn as nn
import torchvision.models as models

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, droprate=0.2, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(droprate)  # Add dropout

        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)

        x = self.inner(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout

        x = self.output_layer(x)
        return x



def make_model(
        learning_rate=0.001,
        size_inner=100,
        droprate=0.2
):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner,
        droprate=droprate
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer


model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

Epoch 1/10
  Train Loss: 1.3570, Train Acc: 0.5580
  Val Loss: 0.8404, Val Acc: 0.7155
Checkpoint saved: mobilenet_v2_01_0.716.pth
Epoch 2/10
  Train Loss: 0.8447, Train Acc: 0.7203
  Val Loss: 0.6509, Val Acc: 0.7771
Checkpoint saved: mobilenet_v2_02_0.777.pth
Epoch 3/10
  Train Loss: 0.6725, Train Acc: 0.7855
  Val Loss: 0.6122, Val Acc: 0.8006
Checkpoint saved: mobilenet_v2_03_0.801.pth
Epoch 4/10
  Train Loss: 0.6137, Train Acc: 0.7943
  Val Loss: 0.6248, Val Acc: 0.7889
Epoch 5/10
  Train Loss: 0.5706, Train Acc: 0.8044
  Val Loss: 0.6295, Val Acc: 0.7742
Epoch 6/10
  Train Loss: 0.5106, Train Acc: 0.8305
  Val Loss: 0.5590, Val Acc: 0.8123
Checkpoint saved: mobilenet_v2_06_0.812.pth
Epoch 7/10
  Train Loss: 0.4623, Train Acc: 0.8432
  Val Loss: 0.5829, Val Acc: 0.7947
Epoch 8/10
  Train Loss: 0.4543, Train Acc: 0.8471
  Val Loss: 0.5988, Val Acc: 0.7859
Epoch 9/10
  Train Loss: 0.4113, Train Acc: 0.8579
  Val Loss: 0.5798, Val Acc: 0.8094
Epoch 10/10
  Train Loss: 0.4129, Train A

In [16]:
# Data Augmentation

# Training transforms WITH augmentation
train_transforms_aug = transforms.Compose([
    transforms.RandomRotation(10),           # Rotate up to 10 degrees
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),  # Zoom
    transforms.RandomHorizontalFlip(),       # Horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation transforms - NO augmentation, same as before
val_transforms_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


# Data Loader

from torch.utils.data import DataLoader

train_dataset_aug = ClothingDataset(
    data_dir='./clothing-dataset-small/train',
    transform=train_transforms_aug
)

val_dataset_aug = ClothingDataset(
    data_dir='./clothing-dataset-small/validation',
    transform=val_transforms_aug
)

print(f"train dataset  Augmented object: {train_dataset_aug}")
print(f"val dataset Augmented object: {val_dataset_aug}")

train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=32, shuffle=False)



def make_model(
        learning_rate=0.001,
        size_inner=100,
        droprate=0.2
):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner,
        droprate=droprate
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer


model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader_aug, val_loader_aug, criterion, num_epochs, device)

train dataset  Augmented object: <__main__.ClothingDataset object at 0x799dd85979d0>
val dataset Augmented object: <__main__.ClothingDataset object at 0x799ddaec6c40>
Epoch 1/10
  Train Loss: 1.3970, Train Acc: 0.5215
  Val Loss: 0.8631, Val Acc: 0.7478
Checkpoint saved: mobilenet_v2_01_0.748.pth
Epoch 2/10
  Train Loss: 0.9001, Train Acc: 0.7005
  Val Loss: 0.6934, Val Acc: 0.7537
Checkpoint saved: mobilenet_v2_02_0.754.pth
Epoch 3/10
  Train Loss: 0.7522, Train Acc: 0.7389
  Val Loss: 0.6069, Val Acc: 0.7889
Checkpoint saved: mobilenet_v2_03_0.789.pth
Epoch 4/10
  Train Loss: 0.6886, Train Acc: 0.7689
  Val Loss: 0.6447, Val Acc: 0.7713
Epoch 5/10
  Train Loss: 0.6109, Train Acc: 0.7924
  Val Loss: 0.5596, Val Acc: 0.8065
Checkpoint saved: mobilenet_v2_05_0.806.pth
Epoch 6/10
  Train Loss: 0.5650, Train Acc: 0.8103
  Val Loss: 0.6069, Val Acc: 0.7889
Epoch 7/10
  Train Loss: 0.5480, Train Acc: 0.8083
  Val Loss: 0.6452, Val Acc: 0.7889
Epoch 8/10
  Train Loss: 0.5182, Train Acc: 0.82

In [24]:
!pip install keras_image_helper onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 21.9 MB/s eta 0:00:00


In [18]:
# Using trained model

# 1.) Loading saved model
# --------------------------------------------------------------------------------------------------------------------------
import glob
# Find best checkpoint
list_of_files = glob.glob('mobilenet_v2_*.pth')
latest_file = max(list_of_files, key=os.path.getctime)
print(f"Loading model from: {latest_file}")
# Load model
model = ClothingClassifierMobileNet(size_inner=100, droprate=0.2, num_classes=10)
model.load_state_dict(torch.load(latest_file))
model.to(device)
model.eval()



# 2.) Making predictions
# ------------------------------------------------------------------------------------------------------------------------------
from keras_image_helper import create_preprocessor
import numpy as np
def preprocess_pytorch_style(X):
    # X: shape (1, 224, 224, 3), dtype=float32, values in [0, 255]
    X = X / 255.0

    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

    # Convert NHWC → NCHW (batch, height, width, channels → batch, channels, height, width)
    X = X.transpose(0, 3, 1, 2)

    # Normalize
    X = (X - mean) / std

    return X.astype(np.float32)

preprocessor = create_preprocessor(preprocess_pytorch_style, target_size=(224, 224))

# Predict from URL
url = 'http://bit.ly/mlbookcamp-pants'
X = preprocessor.from_url(url)
X = torch.Tensor(X).to(device)

with torch.no_grad():
    pred = model(X).cpu().numpy()[0]

classes = [
    "dress", "hat", "longsleeve", "outwear", "pants",
    "shirt", "shoes", "shorts", "skirt", "t-shirt"
]

result = dict(zip(classes, pred.tolist()))
print(result)

Loading model from: mobilenet_v2_08_0.818.pth
{'dress': 0.666660487651825, 'hat': -6.29685115814209, 'longsleeve': -0.6281808018684387, 'outwear': 0.9020641446113586, 'pants': 5.444553852081299, 'shirt': -0.5723286271095276, 'shoes': -5.541243076324463, 'shorts': 0.6568499803543091, 'skirt': -1.7626031637191772, 't-shirt': -3.7652525901794434}


In [25]:
# Exporting model to ONNX

# Create dummy input
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export to ONNX
onnx_path = "clothing_classifier_mobilenet_v2.onnx"


torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    verbose=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model exported to {onnx_path}")

/tmp/ipykernel_416/3547909413.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ClothingClassifierMobileNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ClothingClassifierMobileNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model exported to clothing_classifier_mobilenet_v2.onnx


In [26]:
# Final model whicha has all features
import os
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import torch
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models


input_size = 224

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Transforms
# ---------------------------
# Training transforms WITH augmentation
train_transforms_aug = transforms.Compose([
    transforms.RandomRotation(10),           # Rotate up to 10 degrees
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),  # Zoom
    transforms.RandomHorizontalFlip(),       # Horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation transforms - NO augmentation, same as before
val_transforms_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


# Data Loader
# ------------------------------------
train_dataset_aug = ClothingDataset(
    data_dir='./clothing-dataset-small/train',
    transform=train_transforms_aug
)

val_dataset_aug = ClothingDataset(
    data_dir='./clothing-dataset-small/validation',
    transform=val_transforms_aug
)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=32, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()



class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, droprate=0.2, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')
        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(droprate)  # Add dropout
        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.inner(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.output_layer(x)
        return x



# model checkpointing - saving the best model while training
def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    best_val_accuracy = 0.0  # Initialize variable to track the best validation accuracy

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        # Checkpoint the model if validation accuracy improved
        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            checkpoint_path = f'mobilenet_v2_{epoch+1:02d}_{val_acc:.3f}.pth'
            torch.save(model.state_dict(), checkpoint_path)
            print(f'Checkpoint saved: {checkpoint_path}')



def make_model(learning_rate=0.001, size_inner=100, droprate=0.2):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner,
        droprate=droprate
    )

    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer



# params
best_learning_rate = 0.001
num_epochs = 10

model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader_aug, val_loader_aug, criterion, num_epochs, device)


# TODO
# ---------
# Using trained model
# saving the final model to ONNX format

Epoch 1/10
  Train Loss: 1.4220, Train Acc: 0.5447
  Val Loss: 0.9242, Val Acc: 0.6833
Checkpoint saved: mobilenet_v2_01_0.683.pth
Epoch 2/10
  Train Loss: 0.8673, Train Acc: 0.7115
  Val Loss: 0.7485, Val Acc: 0.7507
Checkpoint saved: mobilenet_v2_02_0.751.pth
Epoch 3/10
  Train Loss: 0.7170, Train Acc: 0.7663
  Val Loss: 0.6400, Val Acc: 0.7830
Checkpoint saved: mobilenet_v2_03_0.783.pth
Epoch 4/10
  Train Loss: 0.6429, Train Acc: 0.7849
  Val Loss: 0.6035, Val Acc: 0.8035
Checkpoint saved: mobilenet_v2_04_0.804.pth
Epoch 5/10
  Train Loss: 0.6329, Train Acc: 0.7836
  Val Loss: 0.6006, Val Acc: 0.7977
Epoch 6/10
  Train Loss: 0.5861, Train Acc: 0.8044
  Val Loss: 0.5666, Val Acc: 0.8065
Checkpoint saved: mobilenet_v2_06_0.806.pth
Epoch 7/10
  Train Loss: 0.5470, Train Acc: 0.8132
  Val Loss: 0.5644, Val Acc: 0.8152
Checkpoint saved: mobilenet_v2_07_0.815.pth
Epoch 8/10
  Train Loss: 0.5079, Train Acc: 0.8256
  Val Loss: 0.6564, Val Acc: 0.7859
Epoch 9/10
  Train Loss: 0.5172, Train A